In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

def create_motion_blur_kernel(size, angle):
    """
    Creates a linear motion blur kernel (PSF) of a given size and angle.
    """
    kernel = np.zeros((size, size))
    center = size // 2
    
    # Calculate line endpoints based on angle
    angle_rad = np.deg2rad(angle)
    x = int(np.cos(angle_rad) * center)
    y = int(np.sin(angle_rad) * center)
    
    # Draw the line on the kernel
    cv2.line(kernel, (center - x, center - y), (center + x, center + y), 1, thickness=1)
    
    # Normalize the kernel so it sums to 1
    kernel = kernel / np.sum(kernel)
    return kernel

def apply_blur_and_noise(image, kernel, noise_var):
    """
    Applies the PSF to the image and adds Gaussian noise.
    """
    blurred = cv2.filter2D(image, -1, kernel)
    
    row, col = image.shape
    mean = 0
    sigma = noise_var ** 0.5
    gauss = np.random.normal(mean, sigma, (row, col))
    
    noisy_blurred = blurred + gauss
    return np.clip(noisy_blurred, 0, 255).astype(np.uint8)

def wiener_deconvolution(blurred_img, kernel, nsr_lambda):
    """
    Applies Wiener Deconvolution in the Frequency Domain.
    """
    h_img, w_img = blurred_img.shape
    h_ker, w_ker = kernel.shape
    
    pad_h = h_img - h_ker
    pad_w = w_img - w_ker
    padded_kernel = np.pad(kernel, ((0, pad_h), (0, pad_w)), 'constant')
    
    G = np.fft.fft2(blurred_img)
    H = np.fft.fft2(padded_kernel)
    
    H_conj = np.conj(H)
    H_squared_mag = np.abs(H) ** 2
    
    W = H_conj / (H_squared_mag + nsr_lambda)
    F_prime = G * W
    
    f_prime_spatial = np.abs(np.fft.ifft2(F_prime))
    
    shift_y, shift_x = h_ker // 2, w_ker // 2
    f_prime_spatial = np.roll(f_prime_spatial, shift_y, axis=0)
    f_prime_spatial = np.roll(f_prime_spatial, shift_x, axis=1)
    
    return np.clip(f_prime_spatial, 0, 255).astype(np.uint8)

def calculate_psnr(img1, img2):
    """
    Calculates the Peak Signal-to-Noise Ratio (PSNR).
    """
    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

# ==========================================
# Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    print("Fetching image from URL...")
    url = "https://d20uiuzezo3er4.cloudfront.net/AI-tools/NumberPlateMasking/before-number.webp"
    
    try:
        # Fetch the image data
        req = urllib.request.urlopen(url)
        arr = np.asarray(bytearray(req.read()), dtype=np.uint8)
        
        # Decode the image directly into grayscale for the deconvolution math
        image = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
        
        if image is None:
            print("Error: Could not decode the image.")
        else:
            print("Image loaded successfully!")
            
            # 2. Parameters for synthetic degradation
            blur_size = 40
            blur_angle = 30  # Degrees
            noise_variance = 20.0
            
            # 3. Create PSF and degrade the image
            psf = create_motion_blur_kernel(blur_size, blur_angle)
            degraded_image = apply_blur_and_noise(image, psf, noise_variance)
            
            # 4. Apply Wiener Deconvolution
            # You can tweak this lambda value to see how it affects noise suppression!
            nsr_lambda = 0.002 
            restored_image = wiener_deconvolution(degraded_image, psf, nsr_lambda)
            
            # 5. Evaluate using PSNR
            psnr_blurred = calculate_psnr(image, degraded_image)
            psnr_restored = calculate_psnr(image, restored_image)
            
            print(f"PSNR (Blurred vs Original):  {psnr_blurred:.2f} dB")
            print(f"PSNR (Restored vs Original): {psnr_restored:.2f} dB")
            
            # 6. Visualization
            plt.figure(figsize=(15, 5))
            
            plt.subplot(1, 3, 1)
            plt.title("Original License Plate")
            plt.imshow(image, cmap='gray')
            plt.axis('off')
            
            plt.subplot(1, 3, 2)
            plt.title(f"Degraded (Blurred + Noise)")
            plt.imshow(degraded_image, cmap='gray')
            plt.axis('off')
            
            plt.subplot(1, 3, 3)
            plt.title(f"Restored (Wiener, NSR={nsr_lambda})")
            plt.imshow(restored_image, cmap='gray')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"An error occurred while fetching the image: {e}")